In [ ]:
DSC630-T302 Predictive Analytics (2263-1)
Assignment 10.2 - Recommender System
Jeremy Marsh

# MovieLens Small Recommender System Using SVD (Surprise)

For this project, I’m building a simple movie recommender system using the MovieLens *ml-latest-small* dataset. My goal is to create a model that can recommend 10 movies based on a single input movie and based on learned patterns in user ratings. I’m using collaborative filtering for this which uses matrix factorization with the SVD algorithm. The SVD algorithm is from the Surprise library.

The workflow I’m following comes from a mix of our readings and online examples. The main ideas are pulled from the following:

- *Applied Predictive Analytics*, Ch. 11  
- “How To Build Your First Recommender System Using Python & MovieLens Dataset” (Medium)  
- “Movie Recommendation System based on MovieLens” (Towards Data Science)  
- Surprise library documentation

These sources describe the same basic pipeline, you load the data, then prepare it for modeling, then train a collaborative filtering model, and then use the learned latent factors to recommend similar items. I’m applying that process step‑by‑step using the MovieLens ratings and movies files.

In [16]:
# importing libraries
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

## Load MovieLens Data

The MovieLens small dataset contains:
- `ratings.csv`: userId, movieId, rating, timestamp  
- `movies.csv`: movieId, title, genres  

Here I will load this data into pandas DataFrames for exploration.


In [4]:
# loading data
ratings = pd.read_csv("ml-latest/ratings.csv")
movies = pd.read_csv("ml-latest/movies.csv")

ratings.head(), movies.head()

(   userId  movieId  rating   timestamp
 0       1        1     4.0  1225734739
 1       1      110     4.0  1225865086
 2       1      158     4.0  1225733503
 3       1      260     4.5  1225735204
 4       1      356     5.0  1225735119,
    movieId                               title  \
 0        1                    Toy Story (1995)   
 1        2                      Jumanji (1995)   
 2        3             Grumpier Old Men (1995)   
 3        4            Waiting to Exhale (1995)   
 4        5  Father of the Bride Part II (1995)   
 
                                         genres  
 0  Adventure|Animation|Children|Comedy|Fantasy  
 1                   Adventure|Children|Fantasy  
 2                               Comedy|Romance  
 3                         Comedy|Drama|Romance  
 4                                       Comedy  )

In [19]:
# basic exploration shapes and unique counts
print("Ratings:", ratings.shape)
print("Movies:", movies.shape)
print("Unique users:", ratings.userId.nunique())
print("Unique movies:", ratings.movieId.nunique())

Ratings: (33832162, 4)
Movies: (86537, 3)
Unique users: 330975
Unique movies: 83239


## Prepare Data for Surprise

The data has to be in a specific format for the Surprise library (a DataFrame with user, item, and rating columns and a `Reader` object defining the rating scale).

In [6]:
# creating the reader object and the dataframe
reader = Reader(rating_scale=(ratings.rating.min(), ratings.rating.max()))
data = Dataset.load_from_df(ratings[['userId','movieId','rating']], reader)

## Train SVD Model

SVD looks for the reasons that explain why people like certain movies. Once it learns those themes for both users and movies, it can predict how someone might rate a movie they haven’t watched yet.

In [9]:
# Splitting the data
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# training the model
svd = SVD(random_state=42)
svd.fit(trainset)

# checking for accuracy
predictions = svd.test(testset)
rmse = accuracy.rmse(predictions)
rmse

RMSE: 0.7862


0.7862074960856344

## Movie-to-Movie Similarity

Once SVD learns a vector for every movie, I measure how similar those vectors are to the user’s movie using cosine similarity.

In [10]:
from numpy.linalg import norm

# a and b are the latent factor vectors for two movies
def cosine_similarity(a, b):
    # denominator is the product of the vector lengths
    denom = norm(a) * norm(b)
    # cosine similarity = dot product divided by vector lengths 
    # if denom is zero, return 0 to avoid dividing by zero
    return np.dot(a, b) / denom if denom else 0

In [11]:
# getting the training set from the SVD model that was trained earlier
trainset = svd.trainset

# Building a lookup using the ID from Suprise and the MovieLens movieId. The raw value is from the MovieLens data and the inner value is what is being used in Surprise.
raw_to_inner = {raw: trainset.to_inner_iid(raw) for raw in trainset._raw2inner_id_items}
inner_to_raw = {inner: trainset.to_raw_iid(inner) for inner in trainset.all_items()}

# Creating table to correlate the movies with the id for quick lookup of recommended movies.
movies_lookup = movies.set_index("movieId")

In [12]:
# This function is to take a movie name from a user input function call to attempt to find 10 similar movies and return their names.

def recommend_similar(movie_title, top_n=10):    
    # Here I use a case insensitive method where any of the part of the movie could be given to return a match and if no match is found it lets the user know.
    matches = movies[movies['title'].str.contains(movie_title, case=False)]
    if matches.empty:
        return f"No movie found for '{movie_title}'"

    # here the movie id is returned of the match if it is in the data
    movie_id = int(matches.iloc[0].movieId)
    if movie_id not in raw_to_inner:
        return "Movie not in training set."

    inner_id = raw_to_inner[movie_id]
    # Get the latent factor vector for the target movie
    target_vec = svd.qi[inner_id]

    sims = []
    # This loops through all of the trained movies
    for other_inner in svd.trainset.all_items():
        # used so it doesn't compare to itself.
        if other_inner == inner_id:
            continue

        # through the loop it compares the cosine similarity between the input movie and the one for this loop
        sim = cosine_similarity(target_vec, svd.qi[other_inner])
        sims.append((other_inner, sim))

    # this sorts the movies based on the similarity score
    sims.sort(key=lambda x: x[1], reverse=True)

    # this keeps just the top number of movies based on the assiogned value in the definition top_n = 10
    top = sims[:top_n]

    recs = []
    # Here the SVD inner ids are converted back to the movie id using the lookup table created earlier and then they are converted to the movie title
    for inner, score in top:
        raw_id = inner_to_raw[inner]
        recs.append((raw_id, movies_lookup.loc[raw_id].title))

    # this just returns a new datafrom with the top 10 recommended movies
    return pd.DataFrame(recs, columns=["movieId","title"])

In [15]:
# here I am running a test with the movie footloose
recommend_similar("Footloose")

,movieId,title
0,1088,Dirty Dancing (1987)
1,2942,Flashdance (1983)
2,597,Pretty Woman (1990)
3,1380,Grease (1978)
4,2144,Sixteen Candles (1984)
5,2145,Pretty in Pink (1986)
6,4132,Mannequin (1987)
7,1968,"Breakfast Club, The (1985)"
8,2420,"Karate Kid, The (1984)"
9,4039,Annie (1982)
